# Úloha 5 - vlastní klasifikátor

Naším úkolem je natrénovat předtreénovaný klasifikátor (z webu [teachablemachine](https://teachablemachine.withgoogle.com/train)), který je schopen z obrázku rozeznat, jaké ovoce se na fotce nachází.

---

1. nasbíráme sady obrázků (alespoň dvě sady, například pomeranče a banány)
1. nátrénujeme model na https://teachablemachine.withgoogle.com/train
1. model exportujeme jako `tensorflow` model
1. nasbíráme testovací sadu obrázků
1. model pomocí python scriptu vyhodnotíme

## Import knihoven

In [ ]:
import os
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
from keras.models import load_model

## Načtení modelu a popisků tříd

1. exportovaný model z Teachable Machine uložte do složky (např. `model/`)
1. načtěte `keras_model.h5` (nebo `model.savedmodel`) a soubor `labels.txt`
1. vypište názvy tříd

In [ ]:
# očekávaná struktura po exportu z Teachable Machine:
# model/
#   keras_model.h5
#   labels.txt
# test/
#   banan_01.jpg
#   pomeranc_01.jpg
#   ...

MODEL_DIR = Path("model")
TEST_DIR = Path("test")

model = load_model(MODEL_DIR / "keras_model.h5", compile=False)
class_names = [line.strip() for line in open(MODEL_DIR / "labels.txt", encoding="utf-8")]

print("Načtené třídy:")
for name in class_names:
    print(" -", name)

## Předzpracování jednoho obrázku

1. načtěte testovací obrázek
1. upravte ho na velikost očekávanou modelem (typicky 224 x 224)
1. normalizujte hodnoty podle dokumentace Teachable Machine
1. vytvořte vstupní pole tvaru `(1, 224, 224, 3)`

In [ ]:
def preprocess_image(path, size=(224, 224)):
    """Úprava obrázku pro Teachable Machine (Image model)."""
    image = Image.open(path).convert("RGB")
    image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

    image_array = np.asarray(image).astype(np.float32)
    # normalizace používaná v oficiálním exportu Teachable Machine
    normalized = (image_array / 127.5) - 1.0

    data = np.ndarray(shape=(1, size[0], size[1], 3), dtype=np.float32)
    data[0] = normalized
    return data


def predict_image(path):
    data = preprocess_image(path)
    prediction = model.predict(data, verbose=0)[0]
    idx = int(np.argmax(prediction))
    return class_names[idx], float(prediction[idx]), prediction

## Predikce a vyhodnocení testovací sady

1. pro každý obrázek v testovací složce spočítejte predikci
1. vypište název třídy a pravděpodobnost
1. (volitelně) spočítejte úspěšnost, pokud znáte správné labely

In [ ]:
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_images = sorted(
    p for p in TEST_DIR.iterdir() if p.suffix.lower() in extensions
) if TEST_DIR.exists() else []

if not test_images:
    print(f"Ve složce '{TEST_DIR}' nebyly nalezeny žádné obrázky.")
    print("Přidejte testovací fotky a spusťte buňku znovu.")
else:
    correct = 0
    total_with_label = 0

    for path in test_images:
        label, confidence, _ = predict_image(path)
        # volitelná kontrola: pokud název souboru obsahuje název třídy
        guessed_ok = any(name.lower().split()[-1] in path.stem.lower() for name in class_names)
        # jednodušší heuristika: začátek názvu souboru = třída (např. banan_01.jpg)
        expected = None
        for name in class_names:
            short = name.split()[-1].lower()
            if path.stem.lower().startswith(short):
                expected = name
                break

        print(f"{path.name:30s} -> {label} ({confidence * 100:.1f} %)")

        if expected is not None:
            total_with_label += 1
            if expected == label:
                correct += 1

    if total_with_label:
        acc = 100.0 * correct / total_with_label
        print(f"\nÚspěšnost na souborech se známým labolem: {correct}/{total_with_label} = {acc:.1f} %")